In [ ]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [ ]:
#creatins session

spark = (( 
          SparkSession
            .builder
            .appName('stock_transaction')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
        .getOrCreate()
        ))

In [ ]:
#load data from table
#creating temporary dataframe

tmp_stocktransaction = (( 
                         spark
                            .read
                            .format('jdbc')
                            .option('driver', mssqldriver)
                            .option('url', mssqlsource)
                            .option('user', mssqluser)
                            .option('password', mssqlpwd)
                            .option('query', 
                                    
                                    """ 
                                        select
                                        
                                            stockitemtransactionid,
                                            stockitemid,
                                            supplierid,
                                            purchaseorderid,
                                            transactiontypeid,
                                            invoiceid,
                                            customerid,
                                            quantity,
                                            lasteditedby
                                            
                                        from warehouse.stockitemtransactions
         
                                    """                        
                                    )                            
                         )).load()

In [ ]:
#rename columns 
renamed_frame = ((tmp_stocktransaction
                    .withColumnRenamed('stockitemtransactionid', 'stocktransactionid')

))

In [ ]:
#remove nulls
removenull = ((
                renamed_frame
                    .select('*', 
                        f.coalesce('supplierid', f.lit(0))
                            .alias('supplierid_n'),
                            
                        f.coalesce('customerid', f.lit(0))
                            .alias('customerid_n'),
                            
                        f.coalesce('purchaseorderid', f.lit(0))
                            .alias('purchaseorderid_n'),
                            
                        f.coalesce('invoiceid', f.lit(0))
                            .alias('invoiceid_n'))        
))

In [ ]:
#drop null columns
#rename normalized columns
dropcolumns = (( removenull
                    .drop('supplierid')
                    .drop('customerid')
                    .drop('purchaseorderid')
                    .drop('invoiceid')
                    .withColumnRenamed('supplierid_n', 'supplierid')
                    .withColumnRenamed('customerid_n', 'customerid')
                    .withColumnRenamed('purchaseorderid_n', 'purchaseorderid')
                    .withColumnRenamed('invoiceid_n', 'invoiceid')
))


In [ ]:
stocktransation = ((dropcolumns
                        .select('stocktransactionid',
                                'stockitemid',
                                'transactiontypeid',
                                'customerid',
                                'invoiceid',
                                'supplierid',
                                'purchaseorderid',
                                'quantity',
                                'lasteditedby')
))
stocktransation.show()

In [12]:
((
stocktransation
    .write
    .format('jdbc')
    .option('url', mssql_dwdest)
    .option('driver', mssqldriver)
    .option('user', mssqluser)
    .option('password', mssqlpwd)
    .option('dbtable', 'fact.stocktransaction')
    .option('mode', 'overwrite')
.save(mode= 'overwrite')
))